# Worksheet 4.2 — Hunting Malicious Network Beacons with Machine Learning

**The scenario:** malware on an infected laptop "phones home" to its command-and-control (C2) server on a timer — every 60 seconds, every 5 minutes, every 15 minutes — to ask *"any instructions for me?"* That regular check-in is called **beaconing**.

**The catch:** lots of *legitimate* software beacons too — EDR heartbeats, NTP time sync, Windows Update, telemetry. A simple "find regular traffic" rule drowns you in false positives. Today we'll see that problem, then train a model that weighs several clues together.

**How to use this notebook**
- Run cells **top to bottom** with **Shift + Enter**. Read the note above each cell first.
- If something breaks: *Kernel → Restart Kernel and Run All Cells*.

**The plan**
1. Get connection logs → 2. Look at what a beacon looks like → 3. Summarise traffic per host-to-destination pair (`groupby`) → 4. Try a simple rule → 5. Train a model → 6. Measure → 7. Build a hunt list → 8. No labels? Try anomaly detection → 9. Run it on your own logs

## Step 0 — Install libraries (run once)

In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib joblib

## Step 1 — Import the tools
No output (apart from the ✅ line) means success.

In [ ]:
import numpy as np                 # number crunching
import pandas as pd                # tables (DataFrames)
import matplotlib.pyplot as plt    # charts

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (precision_score, recall_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)
import joblib

rng = np.random.default_rng(7)     # fixed seed: everyone gets the same data
pd.set_option("display.width", 140)
print("All libraries loaded ✅")

## Step 2 — Generate one day of connection logs

In real life this would be a **Zeek `conn.log`** or **firewall / proxy logs**: one row per network connection.

This cell builds a realistic sample for **200 internal hosts over 24 hours**, containing:

| Traffic | What it looks like | Malicious? |
|---|---|---|
| Normal browsing | bursts of connections, irregular timing, during the workday | No |
| EDR heartbeat, NTP, Windows Update, telemetry | **regular timers**, every host | No |
| Niche business apps on a handful of hosts (licence server, backup agent…) | regular timer, rare destination — **tricky look-alikes!** | No |
| C2 beacons on 40 infected hosts | regular timer with random **jitter**, small same-sized check-ins, mostly rare destinations — but some hide behind **popular cloud/CDN IPs** and some use **heavy jitter** | **Yes** |

> 40 infected hosts out of 200 is far higher than reality — we exaggerate it so the model has enough examples to learn from.

It takes ~10 seconds. You'll see the row count and the first few rows.

In [ ]:
DAY = 24 * 3600                       # seconds in a day
hosts = [f"10.0.{i // 250}.{i % 250 + 10}" for i in range(200)]
rows = []                             # we'll collect one tuple per connection here

def add_beacon(src, dst, port, period, jitter, start, bytes_mean, bytes_sd, bytes_in, spike_rate=0.0):
    """Add connections from src to dst every `period` seconds, +/- jitter."""
    t = start
    while t < DAY:
        b_out = max(40, rng.normal(bytes_mean, bytes_sd))
        if rng.random() < spike_rate:          # occasional bigger upload (tasking / exfil)
            b_out = rng.uniform(5_000, 60_000)
        rows.append((t, src, dst, port, int(b_out), int(max(40, rng.normal(bytes_in, bytes_in * 0.1))),
                     round(rng.uniform(0.05, 0.6), 3)))
        t += period * (1 + rng.uniform(-jitter, jitter))

# --- 1. Normal human browsing: bursty, irregular, mostly 8am-8pm ---
site_pool = [f"{a}.{b}.{c}.{d}" for a, b, c, d in rng.integers(20, 220, size=(600, 4))]
popularity = 1 / np.arange(1, 601) ** 1.1          # a few sites are hugely popular
popularity /= popularity.sum()
for src in hosts:
    for dst in rng.choice(site_pool, size=rng.integers(20, 40), replace=False, p=popularity):
        for _ in range(rng.poisson(3) + 1):          # number of browsing sessions
            t = rng.normal(14 * 3600, 3 * 3600)      # centred on 2pm
            for _ in range(rng.geometric(0.2)):      # connections within the session
                t += rng.exponential(8)
                if 0 < t < DAY:
                    rows.append((t, src, dst, int(rng.choice([443, 443, 443, 80])),
                                 int(rng.lognormal(7, 1.2)), int(rng.lognormal(9, 1.5)),
                                 round(rng.exponential(2), 3)))

# --- 2. Legitimate beacons on every host ---
for src in hosts:
    add_beacon(src, "52.10.40.7", 443, 120, 0.02, rng.uniform(0, 120), 1200, 60, 900)      # EDR heartbeat
    add_beacon(src, "17.253.4.125", 123, 1024, 0.00, rng.uniform(0, 1024), 90, 0, 90)      # NTP time sync
    add_beacon(src, "13.107.4.50", 443, 3600, 0.10, rng.uniform(0, 3600), 2500, 900, 40000) # Windows Update check
    add_beacon(src, "20.42.65.92", 443, 300, 0.05, rng.uniform(0, 300), 3000, 1500, 600)    # Telemetry

# --- 3. Tricky look-alikes: niche business apps on only a few hosts ---
niche_apps = [("185.12.9.33", 600, 0.10, 420),    # licence server
              ("91.198.22.14", 300, 0.25, 650),    # backup agent status
              ("62.4.18.201", 900, 0.15, 380),     # printer fleet monitor
              ("45.77.120.9", 120, 0.35, 520)]     # trading-terminal price feed
for dst, period, jitter, size in niche_apps:
    for src in rng.choice(hosts, rng.integers(3, 9), replace=False):
        add_beacon(src, dst, 443, period, jitter, rng.uniform(0, period), size, size * 0.05, 300)

# --- 4. Malicious C2 beacons on 40 infected hosts ---
c2_servers = [f"{a}.{b}.{c}.{d}" for a, b, c, d in rng.integers(30, 230, size=(15, 4))]
infected = rng.choice(hosts, 40, replace=False)
truth = []                                           # the "answer key" from our incident tickets
for i, src in enumerate(infected):
    if i < 8:   # stealthier malware: C2 hidden behind a popular cloud/CDN IP lots of hosts already use
        dst = str(site_pool[rng.integers(0, 15)])
    else:
        dst = str(rng.choice(c2_servers))
    add_beacon(src, dst, int(rng.choice([443, 443, 80, 8080])),
               period=float(rng.choice([30, 60, 300, 600, 900])),
               jitter=rng.uniform(0.0, 0.5) if i % 5 else rng.uniform(0.5, 0.8),   # some very noisy timers
               start=rng.uniform(0, 8 * 3600),        # infection began overnight or early morning
               bytes_mean=rng.uniform(200, 600), bytes_sd=rng.uniform(5, 40),
               bytes_in=rng.uniform(150, 400), spike_rate=0.02)
    truth.append((src, dst))

conn = pd.DataFrame(rows, columns=["ts", "src_ip", "dst_ip", "dst_port", "bytes_out", "bytes_in", "duration"])
conn = conn.sort_values("ts").reset_index(drop=True)
labels = pd.DataFrame(truth, columns=["src_ip", "dst_ip"]).drop_duplicates().assign(label=1)

print(f"Connections in the log: {len(conn):,}")
print(f"Known malicious host→C2 pairs (answer key): {len(labels)}")
conn.head()

## Step 3 — See the difference with your own eyes

Below, each tick is one connection between one laptop and one destination, zoomed into 8am–2pm.

- **Top:** a human browsing a website — clumps of activity, then silence.
- **Middle:** a C2 beacon — a steady drumbeat, slightly uneven because of jitter, and it carries on overnight when nobody is at the keyboard.
- **Bottom:** a telemetry service — also a steady drumbeat, but **legitimate**. This is why "regular = bad" isn't enough.

In [ ]:
# pick an infected host whose C2 beacon is slow enough to see each tick
gaps = conn.merge(labels, on=["src_ip", "dst_ip"]).groupby(["src_ip", "dst_ip"])["ts"].apply(lambda t: t.diff().mean())
c2_src, c2_dst = gaps.idxmax()
benign = ["52.10.40.7", "17.253.4.125", "13.107.4.50", "20.42.65.92"]
browse = conn[(conn.src_ip == c2_src) & ~conn.dst_ip.isin(benign + [c2_dst])]
top_site = browse["dst_ip"].value_counts().index[0]

examples = [
    ("Human browsing  → " + top_site, conn[(conn.src_ip == c2_src) & (conn.dst_ip == top_site)], "tab:blue"),
    ("C2 beacon       → " + c2_dst,   conn[(conn.src_ip == c2_src) & (conn.dst_ip == c2_dst)],   "tab:red"),
    ("Telemetry (legit) → 20.42.65.92", conn[(conn.src_ip == c2_src) & (conn.dst_ip == "20.42.65.92")], "tab:green"),
]
START, END = 8, 14            # zoom into 8am-2pm so each connection is visible
fig, axes = plt.subplots(3, 1, figsize=(11, 4.5), sharex=True)
for ax, (title, d, color) in zip(axes, examples):
    ax.scatter(d["ts"] / 3600, np.zeros(len(d)), marker="|", s=300, color=color)
    ax.set_yticks([]); ax.set_title(title, loc="left", fontsize=10)
axes[-1].set_xlabel("Hour of day"); axes[-1].set_xlim(START, END)
plt.tight_layout(); plt.show()

## Step 4 — Summarise every host→destination pair with `groupby`

A model can't judge one connection on its own — a beacon is a **pattern over time**. So we group all connections between the same **source host and destination** (like Splunk's `stats ... by src_ip, dst_ip`) and describe each pair with numbers:

| Feature | What it measures | Beacon tends to be… |
|---|---|---|
| `n_conn` | how many connections in the day | high |
| `interval_mean` | average seconds between connections | steady |
| `interval_cv` | **how irregular** the gaps are (spread ÷ average) | **low** = clockwork |
| `bytes_out_mean` | average bytes sent | small check-ins |
| `bytes_out_spread` | how much the typical size varies (ignores the odd big upload) | low = same-sized check-ins |
| `active_hours` | hours between first and last connection | long, including overnight |
| `night_ratio` | share of connections between midnight and 6am | high (nobody's working!) |
| `dst_host_count` | **how many of our hosts talk to this destination** | low = rare, suspicious |

We only keep pairs with **at least 10 connections** — you can't call something a beacon from 3 data points.

In [ ]:
def build_pair_features(conn):
    c = conn.sort_values("ts").copy()
    # seconds since the previous connection between the same pair
    c["interval"] = c.groupby(["src_ip", "dst_ip"])["ts"].diff()
    c["is_night"] = ((c["ts"] % 86400) < 6 * 3600).astype(int)

    pairs = c.groupby(["src_ip", "dst_ip"]).agg(
        n_conn=("ts", "count"),
        first_seen=("ts", "min"),
        last_seen=("ts", "max"),
        interval_mean=("interval", "mean"),
        interval_std=("interval", "std"),
        bytes_out_mean=("bytes_out", "mean"),
        bytes_out_q25=("bytes_out", lambda b: b.quantile(0.25)),
        bytes_out_q75=("bytes_out", lambda b: b.quantile(0.75)),
        bytes_out_median=("bytes_out", "median"),
        night_ratio=("is_night", "mean"),
    ).reset_index()

    pairs["interval_cv"] = pairs["interval_std"] / pairs["interval_mean"]
    # spread of the middle 50% of sizes ÷ typical size (ignores the odd big upload)
    pairs["bytes_out_spread"] = (pairs["bytes_out_q75"] - pairs["bytes_out_q25"]) / pairs["bytes_out_median"]
    pairs["active_hours"] = (pairs["last_seen"] - pairs["first_seen"]) / 3600

    # rarity: how many distinct internal hosts talk to each destination?
    rarity = c.groupby("dst_ip")["src_ip"].nunique().rename("dst_host_count")
    pairs = pairs.merge(rarity, on="dst_ip")

    pairs = pairs[pairs["n_conn"] >= 10].fillna(0).reset_index(drop=True)
    return pairs

FEATURE_COLS = ["n_conn", "interval_mean", "interval_cv", "bytes_out_mean",
                "bytes_out_spread", "active_hours", "night_ratio", "dst_host_count"]

pairs = build_pair_features(conn)
pairs = pairs.merge(labels, on=["src_ip", "dst_ip"], how="left")
pairs["label"] = pairs["label"].fillna(0).astype(int)

print(f"{len(conn):,} connections  →  {len(pairs):,} host→destination pairs to judge")
print(pairs["label"].value_counts().rename({0: "not malicious (0)", 1: "C2 beacon (1)"}))
pairs[["src_ip", "dst_ip"] + FEATURE_COLS + ["label"]].round(2).head(8)

**Sanity check:** the average of each feature for normal pairs vs C2 pairs. Look at `interval_cv`, `night_ratio` and `dst_host_count` especially.

In [ ]:
pairs.groupby("label")[FEATURE_COLS].median().rename(index={0: "not malicious", 1: "C2 beacon"}).round(2)

## Step 5 — First try: a simple hand-written rule

This is the classic beacon-hunting rule many SOCs start with:

> **Alert if a pair has more than 20 connections AND its timing is regular (`interval_cv` < 0.35).**

Let's see how it does against the answer key.

In [ ]:
pairs["rule_alert"] = ((pairs["n_conn"] > 20) & (pairs["interval_cv"] < 0.35)).astype(int)

ConfusionMatrixDisplay.from_predictions(pairs["label"], pairs["rule_alert"],
                                        display_labels=["not malicious", "C2"], cmap="Oranges")
plt.title("Simple rule — confusion matrix"); plt.show()

tn, fp, fn, tp = confusion_matrix(pairs["label"], pairs["rule_alert"]).ravel()
print(f"C2 beacons caught: {tp} of {tp + fn}")
print(f"False positives:   {fp}  ← legitimate beacons (EDR, NTP, updates, telemetry) your analysts would have to triage")
print(f"Precision: {precision_score(pairs['label'], pairs['rule_alert']):.1%}   Recall: {recall_score(pairs['label'], pairs['rule_alert']):.1%}")

**What happened?** The rule catches most C2 — but it also fires on every EDR heartbeat, NTP sync and telemetry beacon. Hundreds of alerts for a few dozen real ones: classic alert fatigue.

An experienced analyst would instantly ask *"but how many of our hosts talk to that destination?"* and *"is it sending the same tiny packet every time?"* — that's exactly what we'll let the model learn.

## Step 6 — Split into training and test sets

`X` = features (evidence), `y` = label (verdict). We hide 30% of pairs as an exam the model never sees during training.

In [ ]:
X = pairs[FEATURE_COLS]
y = pairs["label"]

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, pairs.index, test_size=0.3, random_state=42, stratify=y)

print("Training pairs:", len(X_train), " | C2 among them:", int(y_train.sum()))
print("Test pairs:    ", len(X_test),  " | C2 among them:", int(y_test.sum()))

## Step 7 — Train a Random Forest (create → fit → predict)

`class_weight="balanced"` tells the model that C2 pairs are rare, so missing one should count more — the same instinct as "a missed attack is worse than an extra alert".

In [ ]:
clf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["not malicious", "C2"], cmap="Blues")
plt.title("Random Forest — confusion matrix (test set)"); plt.show()
print(classification_report(y_test, y_pred, target_names=["not malicious", "C2"], digits=3))

## Step 8 — Rule vs model on the same test pairs

A fair comparison: score the simple rule on exactly the same hidden test pairs.

In [ ]:
rule_test = pairs.loc[idx_test, "rule_alert"]

def summarise(name, truth, pred):
    tn, fp, fn, tp = confusion_matrix(truth, pred).ravel()
    return {"approach": name, "C2 caught": f"{tp}/{tp + fn}", "false positives": fp,
            "precision": round(precision_score(truth, pred), 3), "recall": round(recall_score(truth, pred), 3)}

pd.DataFrame([summarise("Simple rule", y_test, rule_test),
              summarise("Random Forest", y_test, y_pred)]).set_index("approach")

## Step 9 — What did the model rely on?

In [ ]:
importance = pd.Series(clf.feature_importances_, index=FEATURE_COLS).sort_values()
importance.plot(kind="barh", color="teal")
plt.title("Feature importance (Random Forest)"); plt.xlabel("Importance"); plt.show()
importance.sort_values(ascending=False).round(3)

## Step 10 — Compare a few models with cross-validation

Five rounds, a different test slice each time, then the average. `F1` blends precision and recall into one number (1.0 = perfect).

In [ ]:
models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000)),
    "Decision Tree":       DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42),
}
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring="f1")
    print(f"{name:20s} F1 = {scores.mean():.3f}  (± {scores.std():.3f})")

## Step 11 — Turn the model into a hunt list

In a SOC you don't want a yes/no — you want a **ranked queue**: "look at these first". `predict_proba` gives each pair a probability of being C2. Here are the top 15 test pairs, most suspicious first, with the evidence an analyst would check.

In [ ]:
hunt = pairs.loc[idx_test, ["src_ip", "dst_ip", "n_conn", "interval_mean", "interval_cv",
                            "bytes_out_mean", "night_ratio", "dst_host_count", "label"]].copy()
hunt["C2_probability"] = clf.predict_proba(X_test)[:, 1]
hunt["actually_C2"] = hunt["label"].map({1: "✔ yes", 0: "no"})
hunt = hunt.drop(columns="label").sort_values("C2_probability", ascending=False)
hunt["interval_mean"] = (hunt["interval_mean"] / 60).round(1)          # show in minutes
hunt = hunt.rename(columns={"interval_mean": "every_x_min"})
hunt.head(15).round(3)

## Step 12 — No labels? Try anomaly detection (Isolation Forest)

In real life you usually **don't have an answer key**. An *unsupervised* model like **Isolation Forest** needs no labels: it flags pairs that look **unusual compared to everything else**. It's a hunting aid, not a verdict machine — let's see how many real C2 pairs land in its top 50.

In [ ]:
iso = IsolationForest(n_estimators=300, contamination="auto", random_state=42)
iso.fit(X)                                         # note: no labels used!
pairs["anomaly_score"] = -iso.score_samples(X)     # higher = more unusual

top50 = pairs.sort_values("anomaly_score", ascending=False).head(50)
print(f"Real C2 pairs in the 50 most unusual: {int(top50['label'].sum())} of {int(pairs['label'].sum())}")
top50[["src_ip", "dst_ip", "n_conn", "interval_cv", "night_ratio", "dst_host_count", "label"]].head(10).round(2)

## Step 13 — Save the model

In [ ]:
joblib.dump({"model": clf, "features": FEATURE_COLS}, "beacon_model.joblib")
print("Saved to beacon_model.joblib")

## Step 14 (optional) — Run it on your own Zeek `conn.log`

Export a day of Zeek connection logs to CSV (from your SIEM or `zeek-cut`), then set `MY_CSV` below. The cell renames Zeek's column names to ours, builds the same features and prints a hunt list. Leave `MY_CSV = None` to skip.

> Only use logs you're authorised to analyse, and keep exports inside approved systems.

In [ ]:
MY_CSV = None     # e.g. "conn_2026-10-27.csv"

if MY_CSV:
    raw = pd.read_csv(MY_CSV)
    raw = raw.rename(columns={"id.orig_h": "src_ip", "id.resp_h": "dst_ip", "id.resp_p": "dst_port",
                              "orig_bytes": "bytes_out", "resp_bytes": "bytes_in"})
    raw["ts"] = pd.to_numeric(raw["ts"], errors="coerce")
    for col in ["bytes_out", "bytes_in", "duration"]:
        raw[col] = pd.to_numeric(raw.get(col, 0), errors="coerce").fillna(0)
    my_pairs = build_pair_features(raw.dropna(subset=["ts"]))
    saved = joblib.load("beacon_model.joblib")
    my_pairs["C2_probability"] = saved["model"].predict_proba(my_pairs[saved["features"]])[:, 1]
    display(my_pairs.sort_values("C2_probability", ascending=False)
            [["src_ip", "dst_ip", "n_conn", "interval_mean", "interval_cv", "dst_host_count", "C2_probability"]].head(25))
else:
    print("Skipped — set MY_CSV to a Zeek conn.log CSV to try it on your own data.")

## What to take away

- **Regular timing alone isn't enough.** Plenty of good software beacons. The simple rule caught C2 but buried it under legitimate heartbeats.
- **Combine clues like an analyst would:** timing regularity + rarity of the destination + same-sized check-ins + overnight activity.
- **A ranked hunt list beats a yes/no alert** — analysts work from the top down.
- **No labels? Start unsupervised**, investigate the top of the list, and turn your verdicts into labels for a supervised model later.

**Experiments to try**
1. In Step 4, remove `dst_host_count` from `FEATURE_COLS` and re-run from there. What happens to false positives?
2. In Step 2, change the C2 jitter to `rng.uniform(0.5, 0.9)` (very noisy malware). Does the rule still work? Does the model?
3. In Step 5, tighten the rule to `interval_cv < 0.1`. What do you gain and lose?
4. Look for the niche business apps (e.g. `185.12.9.33`, `45.77.120.9`) in the full hunt list (`hunt.head(40)`). Did the model fall for the look-alikes?